# Redrob Candidate Ranking, Live Demo

Ranks candidates against the **Senior AI Engineer (Founding Team)** job description and produces a ranked CSV with a reasoning for every pick. Runs on CPU, with no network at rank time.

**To run:** open the **Configuration** cell below, set your repo URL, paste a **Google Drive share link** to the full `candidates.jsonl` into `DRIVE_FILE_URL` (sharing = "Anyone with the link"), then `Runtime -> Run all`. By default it downloads that file and ranks **all candidates** in it. To try it instantly without any Drive file, switch `DATA_SOURCE` to `Bundled sample (50)`. Save the executed notebook with `File -> Save a copy in Drive`.

## How it works

One vectorized pipeline (pandas, numpy, scikit-learn), no per-row loops:

1. **Featurize** each candidate into a single tabular row (skills, career, education, signals).
2. **Flag honeypots**: logically impossible profiles (for example "expert" in a skill used 0 months, or tenure greater than total experience) are detected and excluded from the top results.
3. **Score with three independent lenses**, because each catches what the others miss:
   - **Rules**: a JD-faithful score from config (required skills, experience band, product vs services background, location, availability) minus the penalties the JD spells out (keyword stuffers, CV/speech-only, framework-only, title chasers).
   - **Lexical**: several focused TF-IDF queries for keyword recall.
   - **Semantic**: local MiniLM embeddings that match meaning, catching strong candidates who never write the buzzwords.
4. **Merge with Reciprocal Rank Fusion (RRF)**: combine the three rankings by rank, so no single signal dominates. Weights live in `jd_facets.json`.
5. **Rank and explain**: drop honeypots, then write the top results with evidence-based reasoning.

The job is turned into config once, offline (`canjob/config/jobs/<job_key>/`); at run time the code only reads it.

In [ ]:
#@title Configuration  { display-mode: "form" }
#@markdown ### Code source
REPO_URL = "https://github.com/NavpreetDevpuri/canjob.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

#@markdown ### Candidates
#@markdown Default ranks the **full dataset** from a Google Drive **share link** to `candidates.jsonl` (no mounting needed; set the link's sharing to "Anyone with the link"). Other options: a Drive path you've mounted, the bundled 50-candidate sample, or upload your own (<=100).
DATA_SOURCE = "Google Drive link (full dataset)"  #@param ["Google Drive link (full dataset)", "Mounted Drive path / full dataset", "Bundled sample (50)", "Upload my own"]
#@markdown Paste the shareable link to your `candidates.jsonl` (used when DATA_SOURCE = "Google Drive link").
DRIVE_FILE_URL = ""  #@param {type:"string"}
#@markdown ...or a path to an already-mounted Drive file (used when DATA_SOURCE = "Mounted Drive path").
CANDIDATES_PATH = "/content/drive/MyDrive/candidates.jsonl"  #@param {type:"string"}

#@markdown ### Limits
#@markdown `LIMIT = -1` (default) means ALL candidates in the file. `TOPK` is how many to rank.
LIMIT = -1  #@param {type:"integer"}
TOPK = 100  #@param {type:"integer"}

#@markdown ### Semantic scores
#@markdown ON (default): use the precomputed semantic artifact committed in the repo, so the run needs no torch and no network and is very fast. OFF: recompute embeddings here with MiniLM (installs torch, downloads the model, slower).
USE_PRECOMPUTED = True  #@param {type:"boolean"}

print(f"Data: {DATA_SOURCE} | LIMIT: {LIMIT} | TOPK: {TOPK} | use_precomputed: {USE_PRECOMPUTED}")
if DATA_SOURCE == "Google Drive link (full dataset)" and not DRIVE_FILE_URL.strip():
    print("NOTE: paste your candidates.jsonl share link into DRIVE_FILE_URL above, or switch DATA_SOURCE to 'Bundled sample (50)' to try instantly.")

### 1. Get the code and install dependencies

In [ ]:
import os

if DATA_SOURCE == "Mounted Drive path / full dataset" and CANDIDATES_PATH.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

if "USERNAME/REPO" in REPO_URL:
    raise SystemExit("Set REPO_URL in the Configuration cell to your GitHub repo, then Run all.")

!rm -rf /content/canjob_repo
!git clone --depth 1 --branch $BRANCH $REPO_URL /content/canjob_repo
%cd /content/canjob_repo

!pip install -q -r requirements.txt
if not USE_PRECOMPUTED:
    # Recompute embeddings here instead of using the committed artifact.
    !pip install -q -r requirements-embeddings.txt
    from transformers import AutoModel, AutoTokenizer
    _m = "sentence-transformers/all-MiniLM-L6-v2"
    AutoTokenizer.from_pretrained(_m); AutoModel.from_pretrained(_m)
    print("Embedding model cached for recompute.")

print("Ready in", os.getcwd())

### 2. Resolve the candidate file

In [ ]:
import json

if DATA_SOURCE == "Bundled sample (50)":
    CANDIDATES = "sample/candidates_sample.jsonl"
elif DATA_SOURCE == "Google Drive link (full dataset)":
    # Download the full candidates.jsonl straight from a Drive share link (no mount).
    # The link must be shared as "Anyone with the link". gdown handles big files.
    url = DRIVE_FILE_URL.strip()
    assert url, ("Paste a shareable Google Drive link to candidates.jsonl into DRIVE_FILE_URL "
                 "in the Configuration cell (sharing = 'Anyone with the link').")
    !pip install -q gdown
    import gdown
    CANDIDATES = "/content/candidates.jsonl"
    gdown.download(url=url, output=CANDIDATES, quiet=False, fuzzy=True)
elif DATA_SOURCE == "Upload my own":
    from google.colab import files
    up = files.upload()
    raw = up[list(up)[0]].decode("utf-8", errors="ignore").strip()
    try:
        obj = json.loads(raw)                 # JSON array or single object
        rows = obj if isinstance(obj, list) else [obj]
    except json.JSONDecodeError:
        rows = [json.loads(l) for l in raw.splitlines() if l.strip()]  # JSONL
    rows = rows[:100]
    CANDIDATES = "sample/uploaded.jsonl"
    with open(CANDIDATES, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
else:  # Mounted Drive path / full dataset
    CANDIDATES = CANDIDATES_PATH

assert os.path.exists(CANDIDATES), f"Candidate file not found: {CANDIDATES}"
n_total = sum(1 for _ in open(CANDIDATES))
n_load = n_total if LIMIT < 0 else min(n_total, LIMIT)
print(f"{CANDIDATES}: {n_total} candidates -> loading {n_load}, ranking top {min(n_load, TOPK)}")

### 3. Run the ranker end-to-end

In [ ]:
import time

cli_limit = 0 if LIMIT < 0 else LIMIT     # 0 means all
eff_topk = min(n_load, TOPK)
os.makedirs("output_demo", exist_ok=True)
OUT = "output_demo/submission.csv"

start = time.time()
if not USE_PRECOMPUTED:
    # Regenerate the small semantic artifact from scratch (slower; needs torch).
    !python precompute.py --candidates "$CANDIDATES" --limit $cli_limit
# The ranking step: CPU-only, offline, reads the committed (or just-rebuilt) artifact.
!python rank.py --candidates "$CANDIDATES" --limit $cli_limit --topk $eff_topk --out $OUT
print(f"\nWall time: {time.time()-start:.1f}s")
print("The ranking step (rank.py) is what must finish in 5 min on CPU; it runs offline.")
print("Embedding precompute is a separate one-time step, allowed outside that window.")

### 4. Results

`submission.csv` is the final ranking. Each row has the rank, a **0-1 ensemble fit score** (1.0 = best match to the JD, scores decrease down the ranking), and a reasoning string grounded in the candidate's own profile.

The bar chart below simply visualizes that ensemble fit score for the top candidates, with the best match at the top. A longer bar means a stronger overall fit to the job description.

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 220)
df = pd.read_csv("output_demo/submission.csv")

# quick sanity check that the CSV is well-formed
assert list(df.columns) == ["candidate_id", "rank", "score", "reasoning"]
assert df["rank"].tolist() == list(range(1, len(df) + 1))
assert df["score"].is_monotonic_decreasing and df["candidate_id"].is_unique
print(f"{len(df)} ranked candidates. Ranks 1..N, scores descending, unique ids. Top 15:")
df.head(15)

In [ ]:
import matplotlib.pyplot as plt

top = df.head(min(15, len(df)))
plt.figure(figsize=(9, 0.4 * len(top) + 1))
bars = plt.barh(top["candidate_id"][::-1], top["score"][::-1], color="#2b8cbe")
plt.xlabel("Ensemble fit score  (0 = weakest, 1 = best match to the JD)")
plt.title(f"Top {len(top)} candidates by ensemble fit score (rank 1 at the top)")
plt.xlim(0, 1.08)
for b, v in zip(bars, top["score"][::-1]):
    plt.text(v + 0.01, b.get_y() + b.get_height() / 2, f"{v:.2f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

### 5. Download the ranking (optional)

The ranked CSV is already saved in this session. Download it from the Colab file browser, or uncomment the snippet below to trigger a direct download.

In [ ]:
# Download the ranked CSV (optional).
# The file is saved at output_demo/submission.csv and is also visible in the Colab
# file browser on the left. Auto-download is intentionally disabled; uncomment the
# two lines below if you want the browser to download it directly.
print("Ranked output saved to: output_demo/submission.csv")
# from google.colab import files
# files.download("output_demo/submission.csv")